### Required Codio Assignment 22.4: Fine-Tuning a Pretrained Network

**Expected Time = 60 minutes**

**Total Points = 30**

In addition to the use of a pre-trained network to extract features from a different dataset, the weights can be adjusted or **fine-tuned** as a last step to squeeze additional performance from the network.  To do so, you will again use the `EfficientNetV2B0` network on the `cifar10` data from `keras`.  This time you are encouraged to use the functional API syntax to construct your network.  

For a second example, consult the `keras` documentation example [here](https://keras.io/guides/transfer_learning/).

#### Index

- [Problem 1](#-Problem-1)
- [Problem 2](#-Problem-2)
- [Problem 3](#-Problem-3)

Run the code cell below to import the necessary libraries.

In [2]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2B0
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras import Model, Input

In [3]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
#sub_indices = pd.DataFrame(y_train, columns= ["labels"]).groupby('labels', group_keys=False).apply(lambda x: x.sample(frac=0.3, random_state=123)).index
sub_indices = (
    pd.DataFrame(y_train, columns=["labels"])
    .groupby("labels", group_keys=False)
    .apply(lambda x: x.sample(frac=0.3, random_state=123), include_groups=False)
    .index
)
# Picked up random sub dataset at 30% from the 50K training categorical values and reset X_train and y_train accordingly.
X_train = X_train[sub_indices]
y_train = y_train[sub_indices]

Y_train = to_categorical(y_train)
Y_test = to_categorical(y_test)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


In [4]:
# to_categorical simply transforms the various n categories within the given output variable into n sized arrays for each output .
print(y_test[0:9])
Y_test[0:9]

[[3]
 [8]
 [8]
 [0]
 [6]
 [6]
 [1]
 [6]
 [3]]


array([[0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]])

[Back to top](#-Index)

### Problem 1

#### Training the Network to Convergence

**10 Points**

In the code cell below, use the function `EfficientNetV2B0` with the appropriate input shape and the argument `include_top` equal to `False` to create the model you will be using for this activity. Assign your result to `base_model`.

Use the function `Input()` with argument `shape` equal to `(32, 32, 3)` and assign your result to the variable `inputs`.

Use `base_model` with argument equal to `inputs` and assign your result to the variable `x`.

Use the function `Flatten` to flatten `x`. The pseudocode to complete this step is given below:

```Python
x = Flatten()(...)
```

Pass `x` through a `Dense` layer with 100 hidden nodes and `activation` equal to `relu` and assign the result to the variable `output`. The pseudocode to complete this step is given below:

```Python
output = Dense(..., activation = ...)(...)
```

Use the code `base_model.trainable = False` to ensure that your weights are not trainable.

Use the function `Model()` with argument `inputs` and `output` to define your model. Assign the result to the variable `model`.

Compile `model` using `categorical_crossentropy` as your `loss` and `accuracy` as your `metric`.

Use the `fit()` function on `model` to fit the  training data `X_train` and `Y_train`. Set the argument `validation_data` equal to `(X_test, Y_test)` and the argument `epochs` equal to 1.  Assign the result to the variable `bottom_model` below.

NOTE: This question is computationally expensive, so please be patient with the processing. It may take a few minutes based on your computing power.


In [13]:
### GRADED
base_model = EfficientNetV2B0(include_top=False)
inputs = keras.Input(shape=(32,32,3))
x = base_model(inputs)
x = Flatten()(x)
x = Dense(100, activation="relu")(x)
output = Dense(10, activation="softmax")(x)
base_model.trainable = False
model = Model(inputs, output )
#be sure to compile
model.compile(loss="categorical_crossentropy", metrics=["accuracy"])
print("compiled model...")

# YOUR CODE HERE
bottom_model = model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs =1)
print("model fitted")

### ANSWER CHECK
print(model.summary())

compiled model...
469/469 ━━━━━━━━━━━━━━━━━━━━ 35s 58ms/step - accuracy: 0.4673 - loss: 1.5108 - val_accuracy: 0.5477 - val_loss: 1.2887
model fitted


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_17 (InputLayer)     │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 1, 1, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 100)            │       128,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,177,534 (23.57 MB)

 Trainable params: 129,110 (504.34 KB)

 Non-trainable params: 5,919,312 (22.58 MB)

 Optimizer params: 129,112 (504.35 KB)

None


[Back to top](#-Index)

### Problem 2

#### Setting to Trainable

**10 Points**

In the code cell below, use the code `base_model.trainable = True` to ensure that your weights are now trainable.


Set the final five layers to trainable in the `base_model` as demonstrated in the lectures.

In [14]:
base_model.trainable=True #are the base model weights set to trainable?

In [ ]:
### GRADED
base_model.trainable = True
for layer in base_model.layers:
    #make trainable
    pass

# YOUR CODE HERE


### ANSWER CHECK
for i, layer in enumerate(base_model.layers[-10:]):
    print(f'Layer is trainable: {layer.trainable}')

[Back to top](#-Index)

### Problem 3

#### Refitting the network

**10 Points**

In the code cell below, use the function `compile` on `model` using `categorical_crossentropy` as your `loss` and `accuracy` as your `metric`.

Next, use the `fit()` function on `model` to fit the  training data `X_train` and `Y_train`. Set the argument `validation_data` equal to `(X_test, Y_test)` and the argument `epochs` equal to 1.  Assign the result to the variable `fine_tuned_history` below.


In [ ]:
### GRADED
fine_tuned_history = ''

# YOUR CODE HERE
raise NotImplementedError()

### ANSWER CHECK
fine_tuned_history.history['accuracy'][-1]